In [1]:
!pip install -U langgraph langchain langchain-community==0.4.0 langchain-openai chromadb doppler-sdk pysqlite3-binary ragas datasets


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
import sys
try:
    import pysqlite3 as sqlite3
    sys.modules["sqlite3"] = sqlite3
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError("pysqlite3-binary is required. Run the first cell to install it.") from exc

In [3]:
import init_creds as creds
import sys
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings

sys.path.insert(1, '../../')


AZURE_OPENAI_KEY = creds.get_api_key()
AZURE_OPENAI_ENDPOINT = creds.get_endpoint()
AZURE_OPENAI_API_VERSION = "2025-04-01-preview"

if not AZURE_OPENAI_KEY:
    raise ValueError("No AZURE_OPENAI_KEY set for Azure OpenAI API")
if not AZURE_OPENAI_ENDPOINT:
    raise ValueError("No AZURE_OPENAI_ENDPOINT set for Azure OpenAI API")

llm = AzureChatOpenAI(
    model="gpt-4o-mini",
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_KEY,
    api_version=AZURE_OPENAI_API_VERSION
)

embeddings = AzureOpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=AZURE_OPENAI_KEY,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_version=AZURE_OPENAI_API_VERSION
    )

In [4]:
# from langchain_openai import ChatOpenAI

# llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [5]:

!curl -o manual.txt "https://confident-bucket.s3.us-east-1.amazonaws.com/manual.txt"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 18482  100 18482    0     0  23143      0 --:--:-- --:--:-- --:--:-- 23131


In [6]:
import os
import random
from typing import Annotated, List, TypedDict, Literal
from chromadb.config import Settings
from langchain_community.vectorstores import Chroma


from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from langchain_openai import OpenAIEmbeddings

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from pydantic import BaseModel, Field

# Set API keys
# os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")


class AgentState(TypedDict):
    """State schema for the RAG agent"""

    messages: Annotated[List[BaseMessage], add_messages]
    query: str
    selected_tools: List[str]
    retrieved_context: str
    tool_outputs: List[str]
    next_action: str


# Initialize vector store with your knowledge base
def setup_vector_store():
    """Set up your vector database with documents from local text file"""
    # Load your documents from local text file
    text_file_path = "manual.txt"  # Replace with your actual file path

    try:
        # Load the text file
        loader = TextLoader(text_file_path, encoding="utf-8")
        docs = loader.load()
        

        # Split documents
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=500, chunk_overlap=50
        )
        doc_splits = text_splitter.split_documents(docs)
        # Create vector store
        # embeddings = OpenAIEmbeddings()
        vector_store = Chroma.from_documents(doc_splits, embeddings)

        return vector_store.as_retriever()

    except FileNotFoundError:
        print(
            f"Error: File '{text_file_path}' not found. Please check the file path."
        )
        return None
    except Exception as e:
        print(f"Error loading document: {str(e)}")
        return None


retriever = setup_vector_store()


######## TOOLS ########
@tool
def get_last_day_steps():
    """
    Get the last day's steps from the database
    """
    return random.randint(1000, 5000)


@tool
def get_last_day_average_heart_rate():
    """
    Get the last day's average heart rate from the database
    """
    return random.randint(60, 100)


@tool
def get_last_day_average_sleep_duration_in_hours():
    """
    Get the last day's average sleep duration from the database
    """
    return random.randint(3, 10)


# Tool registry for dynamic selection
tools = [
    get_last_day_steps,
    get_last_day_average_heart_rate,
    get_last_day_average_sleep_duration_in_hours,
]
tool_registry = {tool.name: tool for tool in tools}


######## TYPE DEFINITIONS ########
class RouteQuery(BaseModel):
    """Schema for routing decisions"""

    reasoning: str = Field(description="Reasoning for the routing decision")
    route: Literal["retrieval", "tools", "direct"] = Field(
        description="Where to route the query"
    )
    tools_needed: List[str] = Field(
        description="List of tools needed if route is 'tools'"
    )
    retrieval_query: str = Field(
        description="Optimized query for retrieval if route is 'retrieval'"
    )


def router_node(state: AgentState) -> AgentState:
    """Route the query to appropriate processing path"""

    system_prompt = """You are an intelligent router that decides how to process user queries.
    
    Available options:
    - 'retrieval': Query needs information from the knowledge base
    - 'tools': Query needs external tools (web search, calculations, etc.)  
    - 'direct': Query can be answered directly with general knowledge
    
    Available tools: {tools}
    
    Analyze the user query and decide the best routing approach. If tools are needed,
    specify which ones. If retrieval is needed, optimize the query for better results."""

    user_query = state["messages"][-1].content

    structured_llm = llm.with_structured_output(
        RouteQuery, method="function_calling"
    )

    response = structured_llm.invoke(
        [
            {
                "role": "system",
                "content": system_prompt.format(tools=[t.name for t in tools]),
            },
            {"role": "user", "content": user_query},
        ]
    )

    return {
        "query": user_query,
        "next_action": response.route,
        "selected_tools": response.tools_needed,
        "retrieved_context": (
            response.retrieval_query if response.route == "retrieval" else ""
        ),
    }


def tool_execution_node(state: AgentState) -> AgentState:
    """Execute selected tools"""
    tool_outputs = []
    for tool_name in state["selected_tools"]:
        if tool_name in tool_registry:
            tool = tool_registry[tool_name]
            try:
                # Use the original query for tool execution
                output = tool.invoke({"query": state["query"]})
                tool_outputs.append(f"{tool_name}: {output}")
            except Exception as e:
                tool_outputs.append(f"{tool_name}: Error - {str(e)}")

    return {"tool_outputs": tool_outputs}


def retrieval_node(state: AgentState) -> AgentState:
    """Execute retrieval from vector database"""
    query = state["retrieved_context"] or state["query"]

    try:
        # Retrieve relevant documents
        docs = retriever.invoke(query)
        context = "\n\n".join([doc.page_content for doc in docs])
        return {"retrieved_context": context}
    except Exception as e:
        return {"retrieved_context": f"Retrieval error: {str(e)}"}


def response_synthesis_node(state: AgentState) -> AgentState:
    """Synthesize final response from all available information"""

    # Prepare context from various sources
    context_parts = []

    if state.get("retrieved_context"):
        context_parts.append(
            f"Knowledge Base Context:\n{state['retrieved_context']}"
        )

    if state.get("tool_outputs"):
        tool_context = "\n".join(state["tool_outputs"])
        context_parts.append(f"Tool Outputs:\n{tool_context}")

    context = "\n\n".join(context_parts)

    system_prompt = """You are a helpful assistant that synthesizes information from multiple sources.
    
    Use the provided context to answer the user's question accurately and comprehensively.
    If using information from the context, be sure to reference it appropriately.
    If the context doesn't contain enough information, acknowledge this limitation."""

    messages = [
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": f"Question: {state['query']}\n\nContext:\n{context}",
        },
    ]

    response = llm.invoke(messages)

    return {"messages": [AIMessage(content=response.content)]}


def intial_route_decision(state: AgentState) -> str:
    """Determine next node based on routing decision"""
    next_action = state.get("next_action", "direct")

    if next_action == "tools":
        return "tools"

    if next_action == "retrieval":
        return "retrieval"

    return "retrieval"


def create_rag_graph():
    """Create and compile the RAG workflow graph"""

    # Initialize the state graph
    workflow = StateGraph(AgentState)

    # Add nodes
    workflow.add_node("router", router_node)
    workflow.add_node("retrieval", retrieval_node)
    workflow.add_node("tools", tool_execution_node)
    workflow.add_node("synthesis", response_synthesis_node)

    # define edges
    workflow.add_edge(START, "router")
    workflow.add_conditional_edges(
        "router",
        intial_route_decision,
        {
            "retrieval": "retrieval",
            "tools": "tools",
            "synthesis": "synthesis",
        },
    )

    workflow.add_edge("retrieval", "synthesis")
    workflow.add_edge("tools", "synthesis")
    workflow.add_edge("synthesis", END)

    return workflow.compile()


# Create the graph
app = create_rag_graph()

In [7]:
initial_state = {
    "query": "",
    "selected_tools": [],
    "retrieved_context": "",
    "tool_outputs": [],
    "next_action": "",
}


def _invoke_graph(query: str) -> dict:
    """Helper to invoke the workflow with a fresh state."""

    state = initial_state.copy()
    state["messages"] = [HumanMessage(content=query)]
    state["query"] = query
    return app.invoke(state)


def run_rag_query(query: str) -> str:
    """Run a query through the RAG system and return only the final answer."""

    result = _invoke_graph(query)
    final_message = result["messages"][-1]
    return final_message.content


def run_rag_query_with_context(query: str) -> tuple[str, list[str]]:
    """Run a query and capture retrieved contexts for evaluation."""

    result = _invoke_graph(query)
    final_message = result["messages"][-1].content

    # If the graph didn't carry the context forward, fetch it directly from the retriever
    contexts: list[str] = []
    graph_context = result.get("retrieved_context")
    if graph_context:
        contexts.append(graph_context)
    else:
        try:
            docs = retriever.invoke(result.get("query", query))
            contexts = [doc.page_content for doc in docs]
        except Exception as exc:  # pragma: no cover - defensive guard
            contexts = [f"Retrieval error: {exc}"]

    return final_message, contexts


run_rag_query_with_context("What is the average heart rate of the user?")

('The average heart rate of the user is 83 beats per minute.',
 ["Your resting heart rate, the number of times your heart beats per minute while at rest, is a key indicator of your cardiovascular fitness. A lower resting heart rate generally implies more efficient heart function.[10]\nWhat's a Healthy Range?\nFor most adults, a normal resting heart rate is between 60 and 100 beats per minute (bpm).[10][11][12] Well-trained athletes may have a resting heart rate closer to 40 bpm.[10]\nBest Practices for a Healthy Heart Rate:",
  'Understanding Your Resting Heart Rate (RHR)\nYour RHR is the number of times your heart beats per minute when you are completely at rest.[10] For most adults, a normal RHR falls between 60 and 100 beats per minute (bpm).[11][12] A lower RHR generally signifies a more efficient heart and better cardiovascular fitness.[12][13] Highly trained athletes might even have an RHR around 40 bpm.[12][13]\nFactors That Influence Your Heart Rate:\nYour heart rate can be aff

In [8]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    answer_relevancy,
    context_precision,
    context_recall,
    faithfulness,
)
# from ragas.llms import LangchainLLM
# from ragas.embeddings import LangchainEmbeddings

ragas_llm = llm
ragas_embeddings = embeddings

metrics = [faithfulness, answer_relevancy, context_precision, context_recall]
for metric in metrics:
    if hasattr(metric, "llm"):
        metric.llm = ragas_llm
    if hasattr(metric, "embeddings"):
        metric.embeddings = ragas_embeddings

/tmp/ipykernel_50586/4119965880.py:3: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
/tmp/ipykernel_50586/4119965880.py:3: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
/tmp/ipykernel_50586/4119965880.py:3: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import (
/tmp/ipykernel_50586/4119965880.py:3: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be r

In [9]:
evaluation_samples = [
    {
        "question": "What is the average heart rate of the user?",
        "ground_truth": "The health manual states the user's average heart rate typically ranges between 60 and 100 beats per minute.",
    },
    {
        "question": "How many steps did the user walk yesterday?",
        "ground_truth": "The manual logs daily steps between 1,000 and 5,000 for the tracked user.",
    },
]


def build_ragas_dataset(samples: list[dict]) -> Dataset:
    """Run the agent for each sample and capture results for RAGAS."""

    rows = []
    for sample in samples:
        answer, contexts = run_rag_query_with_context(sample["question"])
        rows.append(
            {
                "question": sample["question"],
                "answer": answer,
                "contexts": contexts or ["No retrieval context captured."],
                "ground_truth": sample["ground_truth"],
            }
        )

    return Dataset.from_list(rows)


ragas_dataset = build_ragas_dataset(evaluation_samples)
ragas_scores = evaluate(
    ragas_dataset,
    metrics=metrics,
    llm=ragas_llm,
    embeddings=ragas_embeddings,
)
ragas_scores

Evaluating:   0%|          | 0/8 [00:00<?, ?it/s]

/usr/local/python/3.12.1/lib/python3.12/site-packages/langchain_openai/chat_models/base.py:549: UserWarning: Unexpected type for token usage: <class 'NoneType'>
  warnings.warn(f"Unexpected type for token usage: {type(new_usage)}")
Exception raised in Job[4]: ValidationError(1 validation error for NLIStatementOutput
  Invalid JSON: expected value at line 1 column 1 [type=json_invalid, input_value='```json\n{\n  "statement...: 0\n    }\n  ]\n}\n```', input_type=TextAccessor]
    For further information visit https://errors.pydantic.dev/2.13/v/json_invalid)


{'faithfulness': 0.0000, 'answer_relevancy': 1.0000, 'context_precision': 0.5000, 'context_recall': 0.5000}

In [10]:
for i in ragas_dataset:
    print("Question:", i["question"])
    print("Answer:", i["answer"])
    print("Ground Truth:", i["ground_truth"])
    print("Contexts:", i["contexts"])
    print("-" * 50)

Question: What is the average heart rate of the user?
Answer: The average heart rate of the user is 63 beats per minute.
Ground Truth: The health manual states the user's average heart rate typically ranges between 60 and 100 beats per minute.
Contexts: ["Your resting heart rate, the number of times your heart beats per minute while at rest, is a key indicator of your cardiovascular fitness. A lower resting heart rate generally implies more efficient heart function.[10]\nWhat's a Healthy Range?\nFor most adults, a normal resting heart rate is between 60 and 100 beats per minute (bpm).[10][11][12] Well-trained athletes may have a resting heart rate closer to 40 bpm.[10]\nBest Practices for a Healthy Heart Rate:", 'Understanding Your Resting Heart Rate (RHR)\nYour RHR is the number of times your heart beats per minute when you are completely at rest.[10] For most adults, a normal RHR falls between 60 and 100 beats per minute (bpm).[11][12] A lower RHR generally signifies a more efficient